In [0]:
"""
Tenday -- Notebook 04: gaming detection (Layer 3).

Two models, deliberately paired:

  IsolationForest   unsupervised anomaly detection. Finds accounts whose
                    behaviour is unlike the bulk of the book. No labels
                    used, so nothing is assumed about what gaming looks
                    like.

  DecisionTree      a SURROGATE trained to reproduce the forest's flags
                    from the same features. The forest decides; the tree
                    explains. Extracting rules from a shallow tree gives
                    the business something it can read, audit and adopt.

Why a surrogate rather than training a classifier on labels
-----------------------------------------------------------
Training directly on `is_gamer` would be supervised detection, which
assumes you already have labelled gamers. In production you do not --
that is the entire problem. The unsupervised path is the honest one, and
ground truth is used only to measure how well it worked.

Contamination
-------------
Set from review capacity, not from the true rate. Setting it to the
known 14.2% would be leaking the answer into the model.
"""

import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

try:
    import mlflow
    MLFLOW = True
except ImportError:
    MLFLOW = False


# Behavioural + redemption features only. No outcome columns
# (net_value, gross_revenue, is_profitable) -- those are what we are
# trying to protect, not inputs to detection.
DETECT_FEATURES = [
    "full_total_spend",
    "full_avg_txn",
    "full_max_txn",
    "full_txn_count",
    "full_category_hhi",
    "full_liquid_share",
    "full_online_share",
    "full_active_day_ratio",
    "full_dormancy_days",
    "full_first30_spend_share",
    "full_last6mo_spend_share",
    "full_distinct_categories",
    "full_spend_velocity",
    "full_spend_per_active_day",
    "full_days_to_qualify",
    "full_redemption_lag",
    "full_qualified",
    "full_redeemed",
]

LOG_FEATURES = [
    "full_total_spend", "full_avg_txn", "full_max_txn",
    "full_txn_count", "full_spend_velocity", "full_spend_per_active_day",
]

# Customers who never qualified have null days_to_qualify. That is not
# missing data -- it is "did not happen". Encoded as the horizon length,
# which is the correct semantic: at least this long.
NEVER_HAPPENED = {
    "full_days_to_qualify": 365.0,
    "full_redemption_lag": 365.0,
}

CONTAMINATION = 0.15   # assumed review capacity, NOT the observed rate


def to_pandas(spark, out, write_format="delta"):
    l3 = spark.read.format(write_format).load(f"{out}/features_l3")
    cols = ["customer_id"] + DETECT_FEATURES
    return l3.select(*cols).toPandas()


def prepare(pdf):
    df = pdf.copy()
    for c, v in NEVER_HAPPENED.items():
        if c in df.columns:
            df[c] = df[c].fillna(v)
    for c in DETECT_FEATURES:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    for c in LOG_FEATURES:
        df[c] = np.log1p(df[c].clip(lower=0))
    return df


def fit_detector(df, contamination=CONTAMINATION, seed=42):
    X = df[DETECT_FEATURES].values
    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X)

    iso = IsolationForest(
        n_estimators=300,
        contamination=contamination,
        max_samples=min(4096, len(Xs)),
        random_state=seed,
        n_jobs=-1,
    ).fit(Xs)

    # sklearn returns -1 for anomalies; flip so 1 = flagged.
    flags = (iso.predict(Xs) == -1).astype(int)
    # score_samples: lower = more anomalous. Negate for an intuitive
    # "higher = riskier" score, then min-max to 0..1.
    raw = -iso.score_samples(Xs)
    score = (raw - raw.min()) / (raw.max() - raw.min())

    out = df[["customer_id"]].copy()
    out["gaming_score"] = score
    out["gaming_flag"] = flags
    return iso, scaler, out


def fit_surrogate(df, flags, max_depth=4, seed=42):
    """
    Approximate the forest's decision with a shallow, readable tree.
    Fidelity (how well the tree reproduces the forest) is reported --
    a surrogate that agrees 60% of the time is not an explanation.
    """
    X = df[DETECT_FEATURES].values
    tree = DecisionTreeClassifier(
        max_depth=max_depth, min_samples_leaf=200,
        class_weight="balanced", random_state=seed,
    ).fit(X, flags)

    fidelity = (tree.predict(X) == flags).mean()
    rules = export_text(tree, feature_names=DETECT_FEATURES, decimals=2)

    importance = (pd.Series(tree.feature_importances_, index=DETECT_FEATURES)
                    .sort_values(ascending=False))
    return tree, fidelity, rules, importance


def evaluate(scored, labels_pdf):
    """
    Ground truth is touched HERE and nowhere else. Detection was fully
    unsupervised; this only measures whether it worked.
    """
    m = scored.merge(labels_pdf, on="customer_id", how="inner")
    y, p, s = m["is_gamer"].values, m["gaming_flag"].values, m["gaming_score"].values

    tn, fp, fn, tp = confusion_matrix(y, p).ravel()
    return {
        "precision": precision_score(y, p, zero_division=0),
        "recall": recall_score(y, p, zero_division=0),
        "f1": f1_score(y, p, zero_division=0),
        "roc_auc": roc_auc_score(y, s),
        "flagged": int(p.sum()),
        "true_gamers": int(y.sum()),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    }


def run(spark, out, write_format="delta", contamination=CONTAMINATION):
    pdf = prepare(to_pandas(spark, out, write_format))

    iso, scaler, scored = fit_detector(pdf, contamination)
    tree, fidelity, rules, importance = fit_surrogate(pdf, scored["gaming_flag"].values)

    labels = (spark.read.format(write_format).load(f"{out}/hidden_labels")
                   .select("customer_id", "is_gamer").toPandas())
    metrics = evaluate(scored, labels)

    print(f"flagged {metrics['flagged']:,} of {len(scored):,} "
          f"({metrics['flagged']/len(scored):.1%})")
    print(f"precision {metrics['precision']:.3f}   "
          f"recall {metrics['recall']:.3f}   "
          f"F1 {metrics['f1']:.3f}   ROC-AUC {metrics['roc_auc']:.3f}")
    print(f"surrogate tree fidelity to forest: {fidelity:.3f}")
    print("\ntop drivers:")
    print(importance.head(8).round(3).to_string())
    print("\n=== extracted rules ===")
    print(rules[:2500])

    if MLFLOW:
        with mlflow.start_run(run_name="isolation_forest", nested=True):
            mlflow.log_param("contamination", contamination)
            mlflow.log_param("n_features", len(DETECT_FEATURES))
            mlflow.log_param("surrogate_depth", 4)
            for k, v in metrics.items():
                mlflow.log_metric(k, v)
            mlflow.log_metric("surrogate_fidelity", fidelity)

    sdf = spark.createDataFrame(scored)
    sdf.write.format(write_format).mode("overwrite").save(f"{out}/gaming_scores")

    return scored, metrics, tree, rules, importance


# ----------------------------------------------------------------------
# Comparison: anomaly detection vs behavioural clustering
# ----------------------------------------------------------------------

def compare_with_segments(spark, out, scored, write_format="delta"):
    """
    The finding this function exists to produce:

    IsolationForest assumes anomalies are FEW and DIFFERENT. Bonus gaming
    is neither -- it is ~14% of the book and internally consistent, which
    is a behavioural MODE, not an outlier. Meanwhile genuinely high-spend
    customers ARE statistical outliers, so the forest flags them too.

    KMeans, which looks for coherent groups rather than isolated points,
    is the better tool for this problem. Reporting that honestly is worth
    more than quietly dropping the model that lost.
    """
    seg = (spark.read.format(write_format).load(f"{out}/segments_named")
                .toPandas())
    lab = (spark.read.format(write_format).load(f"{out}/hidden_labels")
                .select("customer_id", "is_gamer").toPandas())

    m = scored.merge(seg, on="customer_id").merge(lab, on="customer_id")

    # Identify the gaming segment WITHOUT labels: highest mean anomaly
    # score. Ground truth is used only to score the result afterwards.
    gseg = m.groupby("segment")["gaming_score"].mean().idxmax()
    m["segment_flag"] = (m["segment"] == gseg).astype(int)
    m["combined_flag"] = ((m["gaming_flag"] == 1) &
                          (m["segment_flag"] == 1)).astype(int)

    rows = []
    for name, col in [("IsolationForest", "gaming_flag"),
                      ("KMeans segment", "segment_flag"),
                      ("Both (AND)", "combined_flag")]:
        rows.append({
            "method": name,
            "flagged": int(m[col].sum()),
            "precision": round(precision_score(m["is_gamer"], m[col], zero_division=0), 3),
            "recall": round(recall_score(m["is_gamer"], m[col], zero_division=0), 3),
            "f1": round(f1_score(m["is_gamer"], m[col], zero_division=0), 3),
        })

    print(f"gaming segment identified as segment {gseg} "
          f"({m.loc[m.segment == gseg, 'segment_name'].iloc[0]})\n")
    print(pd.DataFrame(rows).to_string(index=False))

    return m, pd.DataFrame(rows)


def precision_at_k(scored, labels_pdf, ks=(0.05, 0.10, 0.15, 0.20)):
    """Ranking quality, independent of where the threshold is set."""
    m = (scored.merge(labels_pdf, on="customer_id")
               .sort_values("gaming_score", ascending=False))
    total = m["is_gamer"].sum()
    rows = []
    for pct in ks:
        head = m.head(int(len(m) * pct))
        rows.append({"top_pct": f"{pct:.0%}",
                     "precision": round(head["is_gamer"].mean(), 3),
                     "recall": round(head["is_gamer"].sum() / total, 3)})
    return pd.DataFrame(rows)

In [0]:
import mlflow
OUT = "/Volumes/workspace/default/tenday"

with mlflow.start_run(run_name="detection"):
    scored, metrics, tree, rules, importance = run(spark, OUT, write_format="delta")

{"ts": "2026-09-09 13:26:59.232", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjEzNTg5NjcxNDM4MzA4NRABIAEyJDAxYTA4NjU4LTUwYzYtNzc2NC1iMjU0LTFjM2M0NDgyMTI3MzokMDU3OTFkNmYtZjFmOS0zNzY5LThlZTItMWY3Y2ExYTM4OTkzSgwIybyF1QYQgMfGuQNQAVgBYAFo57vgspLFow0=.", "context": {}}
{"ts": "2026-09-09 13:26:59.232", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjEzNTg5NjcxNDM4MzA4NRABIAEyJDAxYTA4NjU4LTUwYzYtNzc2NC1iMjU0LTFjM2M0NDgyMTI3MzokMDU3OTFkNmYtZjFmOS0zNzY5LThlZTItMWY3Y2ExYTM4OTkzSgwIybyF1QYQgMfGuQNQAVgBYAFo57vgspLFow0=.", "context": {}}
{"ts": "2026-09-09 13:26:59.232", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjEzNTg5NjcxNDM4MzA4NRABIAEyJDAxYTA4NjU4LTUwYzYtNzc2NC1iMjU0LTFjM2M0NDgyMTI3MzokMDU3OTFkNmYtZjFmOS0zNzY5LThlZTItMWY3Y2ExYTM4OTkz

flagged 30,000 of 200,000 (15.0%)
precision 0.429   recall 0.452   F1 0.440   ROC-AUC 0.860
surrogate tree fidelity to forest: 0.877

top drivers:
full_distinct_categories     0.457
full_spend_per_active_day    0.244
full_days_to_qualify         0.133
full_dormancy_days           0.063
full_spend_velocity          0.033
full_max_txn                 0.030
full_txn_count               0.015
full_active_day_ratio        0.011

=== extracted rules ===
|--- full_distinct_categories <= 8.50
|   |--- full_distinct_categories <= 6.50
|   |   |--- full_txn_count <= 3.96
|   |   |   |--- full_distinct_categories <= 5.50
|   |   |   |   |--- class: 1
|   |   |   |--- full_distinct_categories >  5.50
|   |   |   |   |--- class: 1
|   |   |--- full_txn_count >  3.96
|   |   |   |--- full_liquid_share <= 0.60
|   |   |   |   |--- class: 1
|   |   |   |--- full_liquid_share >  0.60
|   |   |   |   |--- class: 1
|   |--- full_distinct_categories >  6.50
|   |   |--- full_avg_txn <= 7.12
|   |   |   |-

In [0]:
labels = (spark.read.format("delta").load(f"{OUT}/hidden_labels")
               .select("customer_id", "is_gamer").toPandas())

print(precision_at_k(scored, labels).to_string(index=False))
print()
merged, comparison = compare_with_segments(spark, OUT, scored, "delta")

top_pct  precision  recall
     5%      0.498   0.175
    10%      0.462   0.324
    15%      0.429   0.452
    20%      0.407   0.571

gaming segment identified as segment 5 (Minimal Engagers)

         method  flagged  precision  recall    f1
IsolationForest    30000      0.429   0.452 0.440
 KMeans segment     9734      0.008   0.003 0.004
     Both (AND)     5288      0.015   0.003 0.005


In [0]:
import pandas as pd
from pyspark.sql import functions as F
from sklearn.metrics import precision_score, recall_score, f1_score

def identify_gaming_segment(spark, out, write_format="delta"):
    """
    Pick the gaming segment from BEHAVIOUR, not anomaly score.

    Gaming has a definition: qualify fast, redeem fast, front-load spend,
    lean on cash-equivalent categories, then go quiet. We rank segments on
    each of those and take the highest composite. No labels used.
    """
    l3  = spark.read.format(write_format).load(f"{out}/features_l3")
    seg = spark.read.format(write_format).load(f"{out}/segments_named")

    p = (l3.join(seg, "customer_id")
           .groupBy("segment", "segment_name")
           .agg(F.avg("full_days_to_qualify").alias("days_to_qualify"),
                F.avg("full_redemption_lag").alias("redeem_lag"),
                F.avg("full_liquid_share").alias("liquid"),
                F.avg("full_first30_spend_share").alias("front_load"),
                F.avg("full_dormancy_days").alias("dormancy"),
                F.avg("full_qualified").alias("qual_rate"),
                F.count("*").alias("n"))
           .toPandas())

    # Segments that never qualify cannot be gaming an offer they never hit.
    p["days_to_qualify"] = p["days_to_qualify"].fillna(365)
    p["redeem_lag"] = p["redeem_lag"].fillna(365)

    p["gaming_index"] = (
        (1 - p["days_to_qualify"].rank(pct=True))     # faster = higher
      + (1 - p["redeem_lag"].rank(pct=True))
      + p["liquid"].rank(pct=True)
      + p["front_load"].rank(pct=True)
      + p["dormancy"].rank(pct=True)
      + p["qual_rate"].rank(pct=True)
    )

    p = p.sort_values("gaming_index", ascending=False)
    print(p[["segment","segment_name","n","days_to_qualify","redeem_lag",
             "liquid","front_load","dormancy","qual_rate","gaming_index"]]
          .round(3).to_string(index=False))
    return int(p.iloc[0]["segment"]), p


gseg, profile = identify_gaming_segment(spark, OUT)
print(f"\ngaming segment = {gseg}")

# --- rerun the comparison with the correct segment ---
seg = spark.read.format("delta").load(f"{OUT}/segments_named").toPandas()
m = scored.merge(seg, on="customer_id").merge(labels, on="customer_id")
m["segment_flag"]  = (m["segment"] == gseg).astype(int)
m["combined_flag"] = ((m["gaming_flag"] == 1) & (m["segment_flag"] == 1)).astype(int)

rows = []
for name, col in [("IsolationForest", "gaming_flag"),
                  ("KMeans segment",  "segment_flag"),
                  ("Both (AND)",      "combined_flag")]:
    rows.append({"method": name, "flagged": int(m[col].sum()),
                 "precision": round(precision_score(m.is_gamer, m[col], zero_division=0), 3),
                 "recall":    round(recall_score(m.is_gamer, m[col], zero_division=0), 3),
                 "f1":        round(f1_score(m.is_gamer, m[col], zero_division=0), 3)})
print()
print(pd.DataFrame(rows).to_string(index=False))

 segment            segment_name     n  days_to_qualify  redeem_lag  liquid  front_load  dormancy  qual_rate  gaming_index
       4      Bonus Opportunists 27871            9.757       2.494   0.482       0.778   210.050      0.875         5.500
       3 Borderline Opportunists  5040           24.057       4.578   0.256       0.262   100.040      0.049         3.833
       2          Early Churners 27718           96.359      46.066   0.231       0.522   209.082      0.159         3.667
       1         High-Value Core 84517          154.181      45.962   0.230       0.059     1.187      0.928         2.333
       0         Casual Regulars 45120          260.802      43.954   0.231       0.082     6.926      0.139         2.333
       5        Minimal Engagers  9734          365.000     365.000   0.226       0.181    84.354      0.000         1.333

gaming segment = 4

         method  flagged  precision  recall    f1
IsolationForest    30000      0.429   0.452 0.440
 KMeans segment   

## Gaming detection — results

Two unsupervised approaches, evaluated against held-out ground truth
(28,490 true gamers, 14.2% of the book).

| Method | Flagged | Precision | Recall | F1 |
|---|---|---|---|---|
| IsolationForest | 30,000 | 0.429 | 0.452 | 0.440 |
| KMeans segment | 27,871 | **0.998** | **0.977** | **0.987** |
| Both (AND) | 12,301 | 0.997 | 0.431 | 0.601 |

### Why anomaly detection lost

IsolationForest achieved ROC-AUC 0.860 — it ranks risk sensibly — but
only 0.429 precision at a 15% flag rate. The assumption behind
isolation-based methods is that anomalies are **rare and heterogeneous**.
Bonus gaming is neither: it is 14% of the book and behaviourally
consistent. It is a mode, not an outlier.

Meanwhile the customers who genuinely *are* statistical outliers are the
high-value ones — ₹406k average spend, 237 transactions — so the forest
flags them alongside the gamers. Its top drivers were
`distinct_categories` and `spend_per_active_day`: proxies for *unusual*,
not for *gaming*.

The same wrong assumption bit twice. A first attempt to locate the gaming
segment by highest mean anomaly score selected **Minimal Engagers**
(9,734 customers, ₹8,751 spend, 12 transactions) — the most isolated
points in the space precisely because they barely transact. Replacing that
heuristic with a behavioural definition fixed it.

### Identifying the gaming segment without labels

Segments were ranked on six behavioural dimensions consistent with the
definition of gaming — speed to qualify, speed to redeem, cash-equivalent
share, front-loading, dormancy, qualification rate — and the composite
index separated cleanly:

| Segment | Days to qualify | Redeem lag | Liquid | Front-load | Dormancy | Index |
|---|---|---|---|---|---|---|
| Bonus Opportunists | **9.8** | **2.5** | 0.482 | 0.778 | 210 | **5.50** |
| Borderline Opportunists | 24.1 | 4.6 | 0.256 | 0.262 | 100 | 3.83 |
| Early Churners | 96.4 | 46.1 | 0.231 | 0.522 | 209 | 3.67 |
| High-Value Core | 154.2 | 46.0 | 0.230 | 0.059 | 1 | 2.33 |
| Casual Regulars | 260.8 | 44.0 | 0.231 | 0.082 | 7 | 2.33 |
| Minimal Engagers | never | never | 0.226 | 0.181 | 84 | 1.33 |

No labels used. Ground truth was consulted only to score the outcome.

### Ensembling did not help

Requiring both flags held precision at 0.997 but dropped recall to 0.431.
The intersection is bounded by the weaker model, so the ensemble inherits
IsolationForest's blind spots without gaining anything. The segmentation
is used alone.

### Conclusion
For behaviour that is common and internally coherent, clustering is the
right tool and anomaly detection is the wrong one. The negative result is
reported because it is the more useful finding.
